## Delta Lake
> **在大数据的蛮荒时代，分布式存储（如 HDFS、S3）本质上只是一个个“傻大粗”的【普通文件文件夹】。**
> **当你用多个任务往文件夹里并行写 Parquet 文件时，如果没有人管着，就会发生灾难性的物理崩溃：写到一半机器断电了，文件夹里留下一堆残缺的垃圾损坏文件；上游一边在往里写，下游一边在读，下游就会读到一堆残存的“半成品”脏数据。分布式文件夹天生是没有“大管家”的。**
> **Delta Lake 就是为了彻底终结这种无序灾难而诞生的【新时代存储底座】。它在冰冷、没有灵魂的 Parquet 文件之上，强行焊上了一层【智能元数据管理账本（Delta Log）】。它让原本松散的文件文件夹，瞬间具备了像传统关系型数据库（如 MySQL/Oracle）一样强悍、安全、可绝对信任的重工业级统治力。**

今天我们像素级拆开让它名震江湖的四大核心特性。

---

###  一、 什么是 ACID 特性？（重工业生产的安全防线）

在大数据管道中，ACID 决定了你的数据是“坚如磐石”还是“一碰就碎”。它由四个维度的铁律焊死：

#### 1. **A - Atomicity（原子性：要么全成，要么全败）**

* **物理画面**：假设你的任务要往表里写入 1000 万条数据，在写到第 999 万条时，集群的一台机器突然冒烟断电了。
* **传统文件夹的灾难**：那 999 万条残存垃圾会永远留在存储里，污染下游。
* **Delta Lake 的救赎**：元数据账本一看这次写入没成功，**直接拒绝将这次操作写入日志**。在下游看来，这 999 万条数据**在物理上仿佛从未存在过**，表干净得像刚洗过一样。

#### 2. **C - Consistency（一致性：账目永远是对的）**

* 确保在任何并发操作前后，数据的物理状态、索引和账本永远保持完美的闭环，绝不会出现“数据丢了但账本还说在”的灵异事件。

#### 3. **I - Isolation（隔离性：读写互不打扰）**

* **大厂标准痛点**：数仓中台正在往 `bronze_order_items` 表里疯狂写入今天的 1 亿条新数据（写操作需要 10 分钟）。与此同时，运营老板正急着写 SQL 统计昨天的报表（读操作）。
* **Delta Lake 的大招**：**多版本并发控制（MVCC）**。运营老板读的是“上一秒钟账本锁定的旧版本”，中台在暗地里疯狂写写写。两拨大军互不干扰，**下游永远不需要为了等上游写完而痛苦地死锁死等**。

#### 4. **D - Durability（持久性：落袋为安）**

* 一旦分布式事务提交成功，数据就被绝对、永久地固化在云端存储中，哪怕整个机房停电，数据也绝对不会丢失。

---

### 二、 什么是版本控制与时间旅行？（Time Travel）

推荐你死死记住这个核心特性，这是 Delta Lake 最炫酷的“物理后悔药”。

#### 1. 物理本质：每一次修改，都是一次“快照（Snapshot）”

Delta Lake 永远不会物理上去覆盖或修改你的老文件。

* 当你执行 `INSERT`、`UPDATE`、或者 `DELETE` 时，它只是在后台静悄悄地吐出新的 Parquet 文件，并在 `_delta_log/` 文件夹里写下一行全新的 JSON 账本（如 `000001.json`）：*“从这一秒起，老文件的某几行失效，启用新文件的某几行”*。

#### 2. 什么是时间旅行（Time Travel）？

因为老文件的物理肉身和历史账本被完美保留了下来，你可以在代码里**随时穿梭回过去的任意一个历史时刻**！

#####  重工业应用场景：

* **场景 A（灾难恢复）**：今天下午 5 点，一个实习生写错代码，执行了一次错误的 `OVERWRITE`，把整张黄金资产表洗成了一片空白！
* **架构师解药**：别慌，反手一行代码，命令系统直接读取今天下午 4 点 59 分的那个干净版本，一秒钟完美复活全网数据：
```python
# 🚀 穿梭回版本 5（或者指定具体时间戳）
df_resurrected = spark.read.format("delta").option("versionAsOf", 5).load("your_table")

```



---

### 三、 什么是 Schema 约束与演进？（数据防腐阀门）

如果把分布式存储比作一个水库，Schema（表结构：字段名和字段类型）就是进水管上的**高精过滤网**。

#### 1. Schema 约束（Schema Enforcement：严防垃圾入库）

* **物理机制**：你的表原本定义了两个字段：`seller_id (Long)` 和 `order_count (Int)`。
* **灾难发生**：今天上游写前端代码的人手抖，突然往你的管道里塞进来一个长这样的脏数据：`{"seller_id": "ABC", "revenue": 99.9, "new_field": "hacked"}`（类型错了，还多出了乱七八糟的字段）。
* **Delta Lake 的防御**：**直接当场拉响警报，强行报错拦截（Throw Exception）！** 宁可让写入任务崩溃，也绝不允许任何一条污染表结构的脏数据混进你的数仓底座。

#### 2. Schema 演进（Schema Evolution：健康的身体发育）

* **真实业务升级**：公司业务发展了，产品经理说以后订单表确实需要增加一个新列叫 `discount_rate（折扣率）`，这是合法的业务升级。
* **架构师解药**：你不需要像传统 SQL 数据库那样痛苦地去执行 `ALTER TABLE ADD COLUMN` 导致锁表。你只需要在写入时，反手加上一个配置项：
```python
# 🚀 告诉引擎：这是老子特许的业务升级，请在账本里自动把新列加进去！
df_new_version.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("yuto_table")

```




## Experiment
create delta table


In [0]:
columns = ["product_id","product_name","price","category"]

data = [
        (101,"Macbook",3000.0,"Electronics"),
        (102,"Airpods",299.0,"Electronics"),
        (103,"Kobe4",300.0,"Footwear"),
        (104,"Jordan1",200.0,"Footwear")
       ]

df_init = spark.createDataFrame(data,columns)

# write as delta format
df_init.write.format("delta").mode("overwrite").saveAsTable("yutoProduct")

print("success :D")

In [0]:
# read

df_read = spark.read.table("yutoProduct")
display(df_read)

In [0]:
# use sql

spark.sql("select * from yutoProduct where price > 500").show()